<a href="https://colab.research.google.com/github/maabmusa7/BinX_Tech_Internship_Project_GROUP5/blob/ai%2Fwhisperx-pronunciation-scoring/ai-ml/stt-whisperx/pipeline_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pipeline Functions — Deployable STT + Pronunciation Scoring

This notebook wraps the trained V2 model (see notebook 03) into callable
functions ready for integration into the AI service, per the team's
architecture (audio → STT transcript → pronunciation scorer, in parallel
with the LLM reply).

Recap: V2 achieved F1=0.31 on the test set (vs F1=0.25 for the V1 baseline),
using character-level WhisperX alignment features + a lightweight Logistic
Regression classifier. Full details in notebook 03.

In [ ]:
# Cell 2: All imports + device setup (once, at the top)
import whisperx
import torch
import numpy as np
import pandas as pd
import joblib
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
dataset = load_dataset("mispeech/speechocean762")
train_set = dataset["train"]
test_set = dataset["test"]
print("Train:", len(train_set))
print("Test:", len(test_set))

In [ ]:
transcribe_model = whisperx.load_model("small", device, compute_type="float32" if device=="cpu" else "float16")

align_model, align_metadata = whisperx.load_align_model(language_code="en", device=device)

print("WhisperX models loaded!")

2026-09-19 15:46:19 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-09-19 15:46:19 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`


WhisperX models loaded!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

train_char_df = pd.read_csv("/content/drive/MyDrive/train_char_features.csv")
test_char_df = pd.read_csv("/content/drive/MyDrive/test_char_features.csv")

print("Train char features:", train_char_df.shape)
print("Test char features:", test_char_df.shape)

Train char features: (15835, 10)
Test char features: (15928, 10)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

features = ["mean_char_score", "min_char_score", "std_char_score", "char_score_range"]
X_full_train = train_char_df[features]
y_full_train = train_char_df["pronunciation_issue"]

final_model_v2 = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000))
])
final_model_v2.fit(X_full_train, y_full_train)
print("Model retrained from saved features!")

Model retrained from saved features!


In [ ]:
print(transcribe_model)
print(align_model is not None)

True


In [ ]:
def transcribe(audio_array, transcribe_model, language="en"):
    """Converts raw audio to text using WhisperX."""
    result = transcribe_model.transcribe(audio_array, language=language)
    if not result["segments"]:
        return ""
    return " ".join(seg["text"].strip() for seg in result["segments"]).strip()


def extract_char_features(aligned_result):
    """Extracts per-word character-level statistics from WhisperX alignment."""
    chars = aligned_result["segments"][0]["chars"]
    words = []
    current_chars = []

    for item in chars:
        if item["char"] == " ":
            if current_chars:
                scores = [c["score"] for c in current_chars if c.get("score") is not None]
                if scores:
                    words.append({
                        "word": "".join(c["char"] for c in current_chars),
                        "mean_char_score": np.mean(scores),
                        "min_char_score": np.min(scores),
                        "std_char_score": np.std(scores),
                        "char_score_range": np.max(scores) - np.min(scores),
                        "num_chars": len(scores),
                    })
            current_chars = []
        else:
            current_chars.append(item)

    if current_chars:
        scores = [c["score"] for c in current_chars if c.get("score") is not None]
        if scores:
            words.append({
                "word": "".join(c["char"] for c in current_chars),
                "mean_char_score": np.mean(scores),
                "min_char_score": np.min(scores),
                "std_char_score": np.std(scores),
                "char_score_range": np.max(scores) - np.min(scores),
                "num_chars": len(scores),
            })

    return words


CLASSIFIER_THRESHOLD = 0.62  # calibrated on validation set

def score_pronunciation(audio_array, sample_rate, transcript, align_model, align_metadata, classifier, device):
    """Scores pronunciation quality for each word in a transcribed utterance."""
    duration = len(audio_array) / sample_rate
    reference_segments = [{"start": 0.0, "end": duration, "text": transcript}]

    aligned_result = whisperx.align(
        reference_segments, align_model, align_metadata, audio_array, device,
        return_char_alignments=True,
    )

    char_words = extract_char_features(aligned_result)
    if not char_words:
        return {"words": [], "warning": "alignment_failed"}

    feature_cols = ["mean_char_score", "min_char_score", "std_char_score", "char_score_range"]
    X = pd.DataFrame([[w[c] for c in feature_cols] for w in char_words], columns=feature_cols)

    probabilities = classifier.predict_proba(X)[:, 1]

    results = []
    for word_info, prob in zip(char_words, probabilities):
        results.append({
            "word": word_info["word"],
            "confidence": round(float(prob), 3),
            "flag": "needs_attention" if prob >= CLASSIFIER_THRESHOLD else "ok",
        })

    return {"words": results}


def process_audio_turn(audio_array, sample_rate, transcribe_model, align_model, align_metadata, classifier, device):
    """Full AI/ML STT + pronunciation scoring pipeline for one user turn."""
    transcript = transcribe(audio_array, transcribe_model)
    pronunciation_result = score_pronunciation(
        audio_array, sample_rate, transcript, align_model, align_metadata, classifier, device
    )
    return {"transcript": transcript, "pronunciation": pronunciation_result}

In [ ]:
sample = test_set[0]
audio_samples = sample["audio"].get_all_samples()
audio_array = audio_samples.data.numpy().squeeze()
sample_rate = audio_samples.sample_rate

result = process_audio_turn(
    audio_array, sample_rate,
    transcribe_model, align_model, align_metadata, final_model_v2, device
)

print("Generated transcript:", result["transcript"])
print("Reference text (for comparison only):", sample["text"])
print("\nPronunciation result:")
print(result["pronunciation"])

Generated transcript: Mark is going to see elephant.
Reference text (for comparison only): MARK IS GOING TO SEE ELEPHANT

Pronunciation result:
{'words': [{'word': 'Mark', 'confidence': 0.393, 'flag': 'ok'}, {'word': 'is', 'confidence': 0.414, 'flag': 'ok'}, {'word': 'going', 'confidence': 0.345, 'flag': 'ok'}, {'word': 'to', 'confidence': 0.37, 'flag': 'ok'}, {'word': 'see', 'confidence': 0.644, 'flag': 'needs_attention'}, {'word': 'elephant.', 'confidence': 0.402, 'flag': 'ok'}]}


In [ ]:
joblib.dump(final_model_v2, "/content/drive/MyDrive/pronunciation_classifier_v2.pkl")
print("Model saved!")

Model saved!


In [ ]:
test_load = joblib.load("/content/drive/MyDrive/pronunciation_classifier_v2.pkl")
print("Reload test successful:", test_load is not None)

Reload test successful: True
